# KidneyScan AI — KiTS23 training (Kaggle GPU)

Light 3D U-Net for kidney + tumor + cyst segmentation in 3D CT.
Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.

**Setup:** GPU accelerator ON, Internet ON. Free Kaggle GPU, 30 h/week.
**Phase 1:** 20 cases, 30 epochs, ~2-3 GPU hours. **Phase 2:** 400/89 split, 50-100 epochs, ~12-20 GPU hours.

Assistive tool only. Not a diagnosis. A doctor must confirm the result.

In [ ]:
# 1. Check GPU + PyTorch (preinstalled on Kaggle)
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the model code from GitHub (or skip if attached as a dataset)
# Re-running the notebook is safe: clone only if the folder is missing.
!if [ ! -d KidneyScan-AI ]; then git clone --depth 1 https://github.com/Stochastic-Research-Labs/KidneyScan-AI.git; fi
%cd KidneyScan-AI/model
!pip install --quiet nibabel scipy huggingface_hub
!ls -la

In [ ]:
# 3. Download KiTS23 Lite (~40.9 GB) from Hugging Face
# Lite repo is YongchengYAO/KiTS23-Lite. Do NOT use
# neheller/KiTS-Challenge-Imaging here — that is the FULL 94.8 GB set.
# Skip this cell if data is already attached.
# Lite layout is Images/<case>.nii.gz + Masks/<case>.nii.gz;
# train.py reads that layout directly, no conversion step needed.
from huggingface_hub import snapshot_download
snapshot_download(repo_id='YongchengYAO/KiTS23-Lite',
                  repo_type='dataset', local_dir='../data/kits23')
!ls ../data/kits23/Images | head -5
!ls ../data/kits23/Images | wc -l  # expect 489

In [ ]:
# 4. Smoke test: model builds, params in 1-5M, forward pass works
!python light_unet3d.py
!python losses.py

In [ ]:
# 5. PHASE 1 — small test: 20 cases, 30 epochs, ~2-3 GPU hours
# If Dice is near zero here, STOP and fix data/labels first.
!python train.py --data-dir ../data/kits23 --phase phase1 --out-dir checkpoints/phase1

In [ ]:
# 6. PHASE 2 — full train: 400 train / 89 val, 80 epochs
# Run this only after Phase 1 works. Resume with --resume if the session ends.
# !python train.py --data-dir ../data/kits23 --phase phase2 --epochs 80 --out-dir checkpoints/phase2
# Resume example:
# !python train.py --data-dir ../data/kits23 --phase phase2 --epochs 80 --resume checkpoints/phase2/checkpoint_epoch10.pt --out-dir checkpoints/phase2

In [ ]:
# 7. Evaluate on the validation cases (needs predictions first — see inference.py)
# Lite layout: find one real case name first, e.g. !ls ../data/kits23/Images
# then use Images/<case>.nii.gz and Masks/<case>.nii.gz (official layout
# would be case_00000/imaging.nii.gz + case_00000/segmentation.nii.gz).
# !python inference.py ../data/kits23/Images/case_00000.nii.gz --checkpoint checkpoints/phase2/best.pt --ground-truth ../data/kits23/Masks/case_00000.nii.gz --save-mask /tmp/pred.nii.gz
# For full metrics on all val cases, use evaluate.py (see model README).

## After the run

- Download `checkpoints/` from the Kaggle output tab before the session ends.
- Report Dice per class (kidney / tumor / cyst), Surface Dice, HD95 — from `evaluate.py`, never from memory.
- Targets (not results): kidney >0.95, tumor >0.80, cyst >0.65.